<a href="https://colab.research.google.com/github/yianchen903/my-project/blob/main/CG_Minimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def J(x):
    return assemble(1/p*sqrt(inner(grad(x), grad(x)))**p*dx  - f*x*dx)

# Solving by gradient descent

uex = project(uex,V1)
Juex = assemble(1/p*sqrt(inner(grad(uex), grad(uex)))**p*dx  - f*uex*dx)

# u0 = project(Expression("uex ", degree=5,uex=uex), V1)
u0 = project(Expression("1-r ", degree=5,r=r), V1)
# u0 = project(Expression("(x[0]-1)*x[0]*(x[1]-1)*x[1]", degree=5,uex=uex), V1)

eps = 1.0
tol = 1.0e-8
iter = 0
maxiter = 20
t = np.arange(0, 1.1, 0.1)

while eps > tol and iter < maxiter:
  u = Function(V1)
  v = TestFunction(V1)
  alpha = 1


# Find the steepest direction
  w = TrialFunction(V1)

  w = Function(V1)
  uD = Constant(0.0)
  # uD = uex
  bc = DirichletBC(V1, uD, "on_boundary")

  dJdu = pow(sqrt(inner(grad(u0), grad(u0))),p-2)*inner(grad(u0),grad(v))*dx  - f*v*dx
  D = pow(Constant(0.0001) + sqrt(inner(grad(u0), grad(u0))),p-2)*inner(grad(w),grad(v))*dx
  F = D + dJdu

  solve(F == 0  , w, bc)
  w = project(w, V1)
  wnorm = sqrt( assemble(w*w*dx) )

  znn = u0+alpha*w
  Ju0 = assemble(1/p*sqrt(inner(grad(u0), grad(u0)))**p*dx  - f*u0*dx)
  Jw = assemble(1/p*sqrt(inner(grad(w), grad(w)))**p*dx  - f*w*dx)
  Jznn = assemble(1/p*sqrt(inner(grad(znn), grad(znn)))**p*dx  - f*znn*dx)
  print(" ")
  print("iter=%d:" %iter )
  print("Before doing line search :wnorm = %.16f Ju0 = %.16f Jzn = %.16f Jw = %.16f Juex = %.16f " %(wnorm, Ju0, Jznn, Jw, Juex))

# Backtrack line search
  tol_l = 1e-6
  c2 = 1e-4
  c1 = 0.4
  stepsize = alpha
  starting = u0
  direction = w
  Jz0 = assemble(1/p*sqrt(inner(grad(starting), grad(starting)))**p*dx  - f*starting*dx)
  dJdz0 = assemble(pow(sqrt(inner(grad(starting), grad(starting))),p-2)*inner(grad(starting),grad(direction))*dx  - f*direction*dx)
  iter_l = 0
  zn = Function(V1)

  phiv = []
  condv = []
  for i in range(11):
    Phi = J(starting+(i/10.0)*direction)
    cond = J(starting)+c2*(i/10.0)*dJdz0
    phiv.append(Phi)
    condv.append(cond)

  condv = np.array(condv)
  phiv = np.array(phiv)
  plt.plot(t, phiv, marker='o',label='J(z0 + alpha * wn)')
  plt.plot(t, condv, marker='o',label='J(z0) + c2 * alpha * dJdz0')
  plt.xlabel('alpha')
  plt.ylabel('phi(alpha)')
  plt.legend()
  plt.title('phi(alpha) = J(z0 + alpha * wn)')
  plt.grid(True)
  plt.show()


  while stepsize>tol_l:
    print("****line search iter=%d: stepsize=%.16f ****"%(iter_l,stepsize))
    iter_l  = iter_l + 1
    zn = starting+stepsize*direction
    Jzn =  assemble(1/p*sqrt(inner(grad(zn), grad(zn)))**p*dx  - f*zn*dx)

    if Jzn < Jz0+c2*stepsize*dJdz0:
      break
    else:
      stepsize = c1*stepsize



  print("After doing line search: ")
  print("iter_stepsize=%d:" %iter_l)
  print ("stepsize=%.16f Jz0=%.16f Jzn=%.16f RHS=%.16f Juex=%.16f" %(stepsize, Jz0, Jzn, Jz0+c2*stepsize*dJdz0, Juex))


  alpha = stepsize
  u = u0 + alpha * w

  # u_er = project(u-u0, V1)
  u_er = project(w, V1)
  u_exer = project(u-uex, V1)
  u_L2iter = sqrt(assemble(inner(u_er,u_er)*dx))
  uex_L2iter = sqrt(assemble(inner(u_exer,u_exer)*dx))
  eps = u_L2iter
  eps_ex =  uex_L2iter
  print ("error_iter=%.16f error_ex=%.16f" %( eps, eps_ex))
  u0 = u
  iter = iter + 1



u = project(u, V1)
L = project(grad(u), V2)
A = pow(sqrt(inner(grad(u), grad(u))),p-2)*grad(u)
A = project(A, V3)

# plt.figure()
# c1 = plot(L)
# plt.colorbar(c1)
# plt.title('q(grad u) CG solution ')
# plt.legend()
# plt.show()

# plt.figure()
# c2 = plot(A)
# plt.colorbar(c2)
# plt.title('sigma(grad u) CG solution')
# plt.legend()
# plt.show()

plt.figure()
c3 = plot(u)
plt.colorbar(c3)
plt.title('u CG solution')
plt.legend()
plt.show()


uer = project(u-uex, V1)
Ler = project(L-Lex, V2)
Aer = project(A-Aex, V3)

L_L2er = sqrt( assemble(inner(Ler,Ler)*dx) )
A_L2er = sqrt( assemble(inner(Aer,Aer)*dx) )
u_L2er = sqrt( assemble(uer*uer*dx) )



print ("\n")
print("p = %d " %p)
print("u : degree %d , q : degree %d, sigma : degree %d" %(pdeg,pdeg,pdeg))
print("mesh = %d x %d x 2" %(size,size))
print ("|u-uh|_{L2} \t |q-qh|_{L2} \t |sigma-sigma_h|_{L2}")
print ("%.16f \t  %.16f \t  %.16f" % (u_L2er, L_L2er, A_L2er))
print ("\n")
